# Protify on Colab

Trains the RNA → protein converter (`train_rl.py` or `train_supervised.py`) on a Colab runtime.

**Untested on Colab.** The code and the fake task were checked elsewhere, but this notebook (especially the AlphaFold install in step 3) has not been run on a real Colab instance. Start with the smoke test.

How to use it:

1. Runtime → Change runtime type → **T4 GPU** (only needed with `USE_REAL_ALPHAFOLD`; the fake task also runs on CPU).
2. Edit the parameters in step 1. Leave `SMOKE_TEST` on for a first run.
3. Run all cells.

The first real-AlphaFold run also downloads the AlphaFold weights (a few GB) and RNA3DB (1.6 GB, unpacked on first use), which takes several minutes.

Suggested order: (a) `SMOKE_TEST` on, `USE_REAL_ALPHAFOLD` off, to check the code (about a minute). (b) `SMOKE_TEST` on, `USE_REAL_ALPHAFOLD` on, to check the AlphaFold install and see how long one batch takes. (c) Switch `SMOKE_TEST` off for a real run.

In [ ]:
#@title 1. Parameters { display-mode: "form" }
# For a private repo, put a token in the URL: https://<token>@github.com/jhwptsd/Protify.git
REPO_URL = "https://github.com/jhwptsd/Protify.git" #@param {type:"string"}
BRANCH = "claude/modest-ride-tguvbh" #@param {type:"string"}

SMOKE_TEST = True #@param {type:"boolean"}
USE_REAL_ALPHAFOLD = False #@param {type:"boolean"}
TRAINER = "rl" #@param ["rl", "supervised"]

# Used when SMOKE_TEST is off (a smoke test overrides these with small presets)
EPOCHS = 10 #@param {type:"integer"}
BATCH_SIZE = 4 #@param {type:"integer"}
MAX_BATCHES_PER_EPOCH = 0 #@param {type:"integer"}
MAX_SEQ_LEN = 80 #@param {type:"integer"}
NUM_RNAS = 1646 #@param {type:"integer"}
LEARNING_RATE = 0.001 #@param {type:"number"}
ENTROPY_COEF = 0.02 #@param {type:"number"}
NUM_SAMPLES = 4 #@param {type:"integer"}
LOSS = "angle" #@param ["angle", "rmsd", "tm"]
PP_DIST = 0 #@param {type:"number"}
FRESH_START = True #@param {type:"boolean"}
SEED = 0 #@param {type:"integer"}

SAVE_TO_DRIVE = True #@param {type:"boolean"}
DRIVE_FOLDER = "Protify_runs" #@param {type:"string"}

# ---- Smoke-test presets --------------------------------------------------------------------
if SMOKE_TEST:
    if USE_REAL_ALPHAFOLD:
        # ~6 real folds per epoch: checks the install and tells you how long a fold takes
        EPOCHS, BATCH_SIZE, MAX_BATCHES_PER_EPOCH, MAX_SEQ_LEN, NUM_RNAS = 2, 2, 3, 40, 200
    else:
        # fake task: seconds on CPU; checks that the code runs end to end (too short to learn much)
        EPOCHS, BATCH_SIZE, MAX_BATCHES_PER_EPOCH, MAX_SEQ_LEN, NUM_RNAS = 2, 8, 0, 30, 64
    NUM_SAMPLES = min(NUM_SAMPLES, 2)

print("Mode:", "REAL AlphaFold" if USE_REAL_ALPHAFOLD else "fake fold", "| smoke test:" , SMOKE_TEST, "| trainer:", TRAINER)
print(dict(epochs=EPOCHS, batch_size=BATCH_SIZE, max_batches=MAX_BATCHES_PER_EPOCH, max_seq_len=MAX_SEQ_LEN,
           num_rnas=NUM_RNAS, lr=LEARNING_RATE, entropy_coef=ENTROPY_COEF, loss=LOSS))

In [ ]:
#@title 2. Check the runtime
import subprocess, torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "No GPU attached")
if USE_REAL_ALPHAFOLD and not torch.cuda.is_available():
    raise SystemExit("USE_REAL_ALPHAFOLD needs a GPU: Runtime -> Change runtime type -> T4 GPU")

In [ ]:
#@title 3. Get the code and install dependencies
import os, subprocess

def sh(cmd, **kw):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True, **kw)

if not os.path.isdir("/content/Protify"):
    sh(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/Protify")
os.chdir("/content/Protify")

sh("pip install -q --no-warn-conflicts biopython tqdm matplotlib")
if USE_REAL_ALPHAFOLD and not os.path.exists("COLABFOLD_READY"):
    # Same approach as the ColabFold notebooks: keep Colab's own CUDA build of JAX.
    sh('pip install -q --no-warn-conflicts ".[alphafold-minus-jax]"')
    open("COLABFOLD_READY", "w").close()
    print("If the next cells fail with a numpy/jax import error, restart the runtime (Runtime -> Restart) and run again from step 1.")

In [ ]:
#@title 4. Where to keep results
import os, shutil

os.chdir("/content/Protify")
out_dir = "ConverterWeights" if USE_REAL_ALPHAFOLD else "ConverterWeights_fake"
loss_file = {"rl": "losses.txt", "supervised": "losses_sl.txt"}[TRAINER] if USE_REAL_ALPHAFOLD else "losses_fake.txt"

def remove(path):
    if os.path.islink(path) or os.path.isfile(path):
        os.remove(path)
    elif os.path.isdir(path):
        shutil.rmtree(path)

# The repo ships old checkpoints and loss logs; never resume from those by accident.
for path in (out_dir, loss_file):
    remove(path)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    run_name = ("real" if USE_REAL_ALPHAFOLD else "fake") + "_" + TRAINER
    base = f"/content/drive/MyDrive/{DRIVE_FOLDER}/{run_name}"
    os.makedirs(f"{base}/{out_dir}", exist_ok=True)
    # Symlinks, so every epoch's checkpoint and the loss log land on Drive as they are written
    os.symlink(f"{base}/{out_dir}", out_dir)
    os.symlink(f"{base}/{loss_file}", loss_file)
    print("Saving to", base)
    if os.path.exists(f"{base}/{out_dir}/converter.pt") and not FRESH_START:
        print("Found a checkpoint there; training will resume from it.")
else:
    print("Not saving to Drive: results are lost when the runtime disconnects.")

In [ ]:
#@title 5. Train
import subprocess, sys, time

script = "train_rl.py" if TRAINER == "rl" else "train_supervised.py"
cmd = [sys.executable, "-u", script,
       "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE), "--max-seq-len", str(MAX_SEQ_LEN),
       "--num-rnas", str(NUM_RNAS), "--lr", str(LEARNING_RATE), "--loss", LOSS, "--seed", str(SEED)]
if TRAINER == "rl":
    cmd += ["--entropy-coef", str(ENTROPY_COEF)]
else:
    cmd += ["--num-samples", str(NUM_SAMPLES)]
if MAX_BATCHES_PER_EPOCH:
    cmd += ["--max-batches", str(MAX_BATCHES_PER_EPOCH)]
if PP_DIST:
    cmd += ["--pp-dist", str(PP_DIST)]
if FRESH_START:
    cmd += ["--fresh"]
if not USE_REAL_ALPHAFOLD:
    cmd += ["--fake-fold"]

print(" ".join(cmd), "\n")
start = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
elapsed = time.time() - start
print(f"\nExit code {proc.returncode}; took {elapsed/60:.1f} minutes")
if proc.returncode != 0:
    raise RuntimeError("Training failed; see the output above")

In [ ]:
#@title 6. Results
import os
import matplotlib.pyplot as plt

rows = [l.split("\t") for l in open(loss_file) if l.strip()] if os.path.exists(loss_file) else []
if not rows:
    print("No batches were logged")
else:
    loss = [float(r[0]) for r in rows]
    n = len(loss)
    print(f"{n} batches logged; first-5 mean loss {sum(loss[:5])/min(5, n):.3f}, last-5 mean loss {sum(loss[-5:])/min(5, n):.3f}")
    if USE_REAL_ALPHAFOLD:
        print(f"About {elapsed/n:.0f} s per batch of {BATCH_SIZE} on this run. That figure includes the one-off downloads "
              f"(AlphaFold weights, RNA3DB), so on a smoke test it overstates the steady-state time: run again to measure it.")
    fig, ax = plt.subplots()
    ax.plot(loss)
    ax.set_xlabel("Batch"); ax.set_ylabel("Loss"); ax.grid()
    plt.show()
    if len(rows[0]) > 2:
        print("Final policy entropy:", float(rows[-1][2]), "(3.0 = uniformly random; near 0 = decided)")
print("Checkpoints:", sorted(os.listdir(out_dir)) if os.path.isdir(out_dir) else "none")

In [ ]:
#@title 7. Try the trained converter on some RNA sequences
import subprocess, sys
result = subprocess.run([sys.executable, "run_converter.py", "GGUAUGAGGGUAUU", "AAAAAAAAAA", "ACGUACGUACGU",
                        "--checkpoint", f"{out_dir}/converter.pt"], capture_output=True, text=True)
print(result.stdout or result.stderr)

## Reading the smoke-test result

* **Fake task, smoke test:** it only needs to finish without errors and write a checkpoint. Two epochs are far too short for the loss to fall; a full fake run needs roughly 800 batches (`SMOKE_TEST` off, `EPOCHS` 40, `BATCH_SIZE` 16, `NUM_RNAS` 320, `MAX_SEQ_LEN` 30), and then the loss should reach about 0.
* **Real AlphaFold, smoke test:** it should finish and print a seconds-per-batch figure. If it fails during the AlphaFold install or import, that is the part most likely to need adjusting for the current Colab image.
* **Real run:** compare the loss against random sequences first. If the trained loss never drops below the random level, the reward signal is the problem, not the code.